In [3]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import pynapple as nap
from scipy.ndimage import gaussian_filter, rotate
from scipy.signal import correlate2d
from tqdm import tqdm

from Utils.json_tools import read_formatted_json
from Utils.kilosort_utils import (
    convert_time_list_to_nap_tsd,
    locate_spikes,
)
from Utils.load_files import get_interval_pairs
from Utils.tuning_curve_utils import get_exposure_timestamps

file_names = read_formatted_json("./file_names.json")
session_info_filename: str = file_names["session_info_filename"]
interval_table_filename: str = file_names["interval_table_filename"]


32


## Analysis configuration

These values are used only to generate per-recording caches. The plotting
notebook has a separate configuration cell.


In [4]:
recording_root = Path(r"/mnt/senzailab/Kai/#Recording/m19")
date: str | int = "260831"
analyzeRecList = [2]
probe_name: str = "A"
phase_key: str = "baseline"
poseConfig = {
    2: ("260831.csv", "flat"),
}

arena_range = [(365, 925), (205, 770)]
spatial_bins: int = 40
smooth_sigma: float = 1.5
min_occupancy_s: float = 0.1
min_spikes: int = 100
presence_bin_s: float = 60
min_presence_ratio: float = 0.5
min_spike_time_coverage: float = 0.9

num_shuffle: int = 500
shuffle_seed: int = 0
min_shift_s: float = 20
spatial_alpha: float = 0.01
min_map_stability: float = 0.3
min_grid_score: float = 0.3
min_border_score: float = 0.5

egocentric_angle_bin_deg: float = 6
egocentric_num_distance_bins: int = 20
egocentric_smooth_sigma_bins: float = 5
egocentric_min_occupancy_s: float = 0.0
egocentric_num_shuffle: int = 100
egocentric_shuffle_seed: int = 0
egocentric_min_shift_s: float = 30
egocentric_alpha: float = 0.05

date_dir = recording_root / str(date)


In [1]:
def session_paths(num_of_rec):
    session_dir = date_dir / f"{date}_{num_of_rec}"
    data_dir = session_dir / "data"
    kilosort_dir = next(
        (session_dir / "kilosort" / f"Probe{probe_name}").glob(
            "kilosort_*"
        )
    )
    return data_dir, kilosort_dir


def result_paths(num_of_rec):
    data_dir, _ = session_paths(num_of_rec)
    summary_path = (
        data_dir / f"spatial_cells_rec{num_of_rec}_Probe{probe_name}.csv"
    )
    spatial_shuffle_path = data_dir / (
        f"spatial_shuffle_rec{num_of_rec}_{phase_key}_Probe{probe_name}_"
        f"n{num_shuffle}_seed{shuffle_seed}_shift{min_shift_s:g}.npz"
    )
    egocentric_shuffle_path = data_dir / (
        f"egocentric_shuffle_rec{num_of_rec}_{phase_key}_"
        f"Probe{probe_name}_n{egocentric_num_shuffle}_"
        f"seed{egocentric_shuffle_seed}_"
        f"shift{egocentric_min_shift_s:g}.npz"
    )
    map_cache_path = data_dir / (
        f"spatial_maps_rec{num_of_rec}_{phase_key}_Probe{probe_name}.npz"
    )
    return (
        summary_path,
        spatial_shuffle_path,
        egocentric_shuffle_path,
        map_cache_path,
    )


def read_pose(pose_path, pose_format):
    if pose_format == "motive":
        pose = pd.read_csv(
            pose_path,
            skiprows=8,
            header=None,
            usecols=[0, 5, 6, 8, 9, 11, 12],
            names=[
                "frame",
                "center_x",
                "center_y",
                "front_x",
                "front_y",
                "back_x",
                "back_y",
            ],
        )
        pose["hd_deg"] = np.degrees(
            np.arctan2(
                pose["back_x"] - pose["front_x"],
                pose["back_y"] - pose["front_y"],
            )
        )
    else:
        pose = pd.read_csv(pose_path)
        if "hd_deg" not in pose:
            pose["hd_deg"] = np.degrees(
                np.arctan2(
                    pose["back_x"] - pose["front_x"],
                    pose["back_y"] - pose["front_y"],
                )
            )
        pose = pose[["frame", "center_x", "center_y", "hd_deg"]]

    required = ["frame", "center_x", "center_y", "hd_deg"]
    pose = (
        pose.dropna(subset=required)
        .sort_values("frame")
        .drop_duplicates("frame")
    )
    pose["frame"] = pose["frame"].astype(int)
    pose["hd_deg"] = (pose["hd_deg"] + 180) % 360 - 180
    return pose.reset_index(drop=True)


## Allocentric spatial-cell helpers


In [6]:
def smooth_rate_maps(rate_maps, sigma, min_occupancy_s):
    occupancy_s = (
        np.asarray(rate_maps.attrs["occupancy"])
        / rate_maps.attrs["fs"]
    )
    spike_counts = np.nan_to_num(rate_maps.values) * occupancy_s
    numerator = gaussian_filter(
        spike_counts,
        sigma=(0, sigma, sigma),
    )
    denominator = gaussian_filter(occupancy_s, sigma=sigma)
    smoothed = rate_maps.copy()
    smoothed.values = np.full_like(rate_maps.values, np.nan)
    np.divide(
        numerator,
        denominator,
        out=smoothed.values,
        where=denominator > 0,
    )
    smoothed.values[:, occupancy_s < min_occupancy_s] = np.nan
    return smoothed


def map_correlation(first, second):
    valid = np.isfinite(first) & np.isfinite(second)
    if valid.sum() < 2:
        return np.nan
    return np.corrcoef(first[valid], second[valid])[0, 1]


def spatial_autocorrelation(rate_map):
    valid = np.isfinite(rate_map).astype(float)
    values = np.nan_to_num(rate_map)
    pairs = correlate2d(valid, valid, mode="full")
    pairs_safe = np.maximum(pairs, 1)

    sum_a = correlate2d(values, valid, mode="full")
    sum_b = correlate2d(valid, values, mode="full")
    sum_ab = correlate2d(values, values, mode="full")
    sum_aa = correlate2d(values ** 2, valid, mode="full")
    sum_bb = correlate2d(valid, values ** 2, mode="full")

    covariance = sum_ab - sum_a * sum_b / pairs_safe
    variance_a = sum_aa - sum_a ** 2 / pairs_safe
    variance_b = sum_bb - sum_b ** 2 / pairs_safe
    denominator = np.sqrt(np.maximum(variance_a * variance_b, 0))

    autocorrelation = np.full_like(covariance, np.nan)
    np.divide(
        covariance,
        denominator,
        out=autocorrelation,
        where=(pairs >= 20) & (denominator > 0),
    )
    return autocorrelation


def grid_score(autocorrelation):
    y, x = np.indices(autocorrelation.shape)
    center = (np.asarray(autocorrelation.shape) - 1) / 2
    radius = np.hypot(x - center[1], y - center[0])
    annulus = (
        (radius >= 0.12 * min(autocorrelation.shape))
        & (radius <= 0.40 * min(autocorrelation.shape))
    )

    correlations = {}
    for angle in (30, 60, 90, 120, 150):
        rotated = rotate(
            autocorrelation,
            angle,
            reshape=False,
            order=1,
            mode="constant",
            cval=np.nan,
        )
        valid = annulus & np.isfinite(autocorrelation) & np.isfinite(rotated)
        correlations[angle] = (
            np.corrcoef(autocorrelation[valid], rotated[valid])[0, 1]
            if valid.sum() >= 2
            else np.nan
        )

    return min(correlations[60], correlations[120]) - max(
        correlations[30],
        correlations[90],
        correlations[150],
    )


def border_score(rate_map):
    if not np.isfinite(rate_map).any() or np.nanmax(rate_map) <= 0:
        return np.nan, "none"
    field = rate_map > 0.2 * np.nanmax(rate_map)
    wall_width = 2
    coverage = {
        "top": np.any(field[:wall_width], axis=0).mean(),
        "bottom": np.any(field[-wall_width:], axis=0).mean(),
        "left": np.any(field[:, :wall_width], axis=1).mean(),
        "right": np.any(field[:, -wall_width:], axis=1).mean(),
    }
    wall = max(coverage, key=coverage.get)

    y, x = np.indices(rate_map.shape)
    distance_to_wall = np.minimum.reduce(
        [x, rate_map.shape[1] - 1 - x, y, rate_map.shape[0] - 1 - y]
    )
    denominator = np.nansum(rate_map)
    if denominator <= 0:
        return np.nan, wall
    mean_distance = np.nansum(rate_map * distance_to_wall) / denominator
    normalized_distance = mean_distance / (min(rate_map.shape) / 2)
    score_denominator = coverage[wall] + normalized_distance
    if score_denominator == 0:
        return np.nan, wall
    score = (coverage[wall] - normalized_distance) / score_denominator
    return score, wall


## Egocentric boundary-cell helpers

The rectangular arena is ray-cast from each tracked head position. The
head-direction convention used by the pose files is preserved: 0 degrees is
image-up/front and positive angles point left. Occupancy and spike maps are
smoothed jointly with circular wrapping along the angle axis.


In [7]:
def egocentric_distance_edges(arena_range, num_bins):
    width = arena_range[0][1] - arena_range[0][0]
    height = arena_range[1][1] - arena_range[1][0]
    max_distance = 0.5 * min(width, height)
    if max_distance <= 1:
        raise ValueError("arena must span more than two coordinate units")
    return np.r_[0.0, np.geomspace(1.0, max_distance, num_bins)]


def egocentric_distance_centers(distance_edges):
    centers = np.empty(len(distance_edges) - 1, dtype=float)
    centers[0] = distance_edges[1] / 2
    centers[1:] = np.sqrt(distance_edges[1:-1] * distance_edges[2:])
    return centers


def raycast_rectangle_distances(
    position_xy,
    head_direction_deg,
    egocentric_angle_deg,
    arena_range,
):
    position_xy = np.asarray(position_xy, dtype=float)
    head_direction_deg = np.asarray(head_direction_deg, dtype=float)
    absolute_angle = np.deg2rad(
        head_direction_deg[:, None] + egocentric_angle_deg[None, :]
    )

    # In image coordinates y grows downward. With this convention, 0 is
    # forward and positive egocentric angles point to the animal's left.
    direction_x = -np.sin(absolute_angle)
    direction_y = -np.cos(absolute_angle)
    x = position_xy[:, 0, None]
    y = position_xy[:, 1, None]
    x_min, x_max = arena_range[0]
    y_min, y_max = arena_range[1]

    with np.errstate(divide="ignore", invalid="ignore"):
        distance_x = np.where(
            direction_x > 0,
            (x_max - x) / direction_x,
            (x_min - x) / direction_x,
        )
        distance_y = np.where(
            direction_y > 0,
            (y_max - y) / direction_y,
            (y_min - y) / direction_y,
        )
    distance = np.minimum(distance_x, distance_y)

    inside = (
        (position_xy[:, 0] >= x_min)
        & (position_xy[:, 0] <= x_max)
        & (position_xy[:, 1] >= y_min)
        & (position_xy[:, 1] <= y_max)
    )
    distance[~inside] = np.nan
    distance[(distance < 0) | ~np.isfinite(distance)] = np.nan
    return distance


def egocentric_occupancy(distance_bin_index, frame_dt_s, num_distance_bins):
    occupancy_s = np.zeros(
        (num_distance_bins, distance_bin_index.shape[1]),
        dtype=float,
    )
    for angle_index in range(distance_bin_index.shape[1]):
        bins = distance_bin_index[:, angle_index]
        valid = bins >= 0
        occupancy_s[:, angle_index] = np.bincount(
            bins[valid],
            weights=np.full(valid.sum(), frame_dt_s),
            minlength=num_distance_bins,
        )
    return occupancy_s


def prepare_egocentric_geometry(
    pose_times,
    position_xy,
    head_direction_deg,
    interval_pair,
    arena_range,
    angle_bin_deg,
    num_distance_bins,
):
    start, end = interval_pair
    pose_times = np.asarray(pose_times, dtype=float)
    position_xy = np.asarray(position_xy, dtype=float)
    head_direction_deg = np.asarray(head_direction_deg, dtype=float)
    valid = (
        (pose_times >= start)
        & (pose_times <= end)
        & np.isfinite(position_xy).all(axis=1)
        & np.isfinite(head_direction_deg)
    )
    frame_times = pose_times[valid]
    frame_position = position_xy[valid]
    frame_head_direction = head_direction_deg[valid]
    if len(frame_times) < 2:
        raise ValueError("not enough valid pose frames for egocentric analysis")

    positive_steps = np.diff(frame_times)
    positive_steps = positive_steps[positive_steps > 0]
    frame_dt_s = float(np.median(positive_steps))
    angle_edges_deg = np.arange(
        -180,
        180 + angle_bin_deg / 2,
        angle_bin_deg,
        dtype=float,
    )
    angle_centers_deg = (
        angle_edges_deg[:-1] + angle_edges_deg[1:]
    ) / 2
    distance_edges_px = egocentric_distance_edges(
        arena_range,
        num_distance_bins,
    )
    distance_centers_px = egocentric_distance_centers(distance_edges_px)
    wall_distance = raycast_rectangle_distances(
        frame_position,
        frame_head_direction,
        angle_centers_deg,
        arena_range,
    )
    distance_bin_index = np.searchsorted(
        distance_edges_px,
        wall_distance,
        side="right",
    ) - 1
    distance_bin_index[
        ~np.isfinite(wall_distance)
        | (wall_distance < distance_edges_px[0])
        | (wall_distance >= distance_edges_px[-1])
    ] = -1
    distance_bin_index = distance_bin_index.astype(np.int16)
    occupancy_s = egocentric_occupancy(
        distance_bin_index,
        frame_dt_s,
        num_distance_bins,
    )
    return {
        "frame_times": frame_times,
        "frame_dt_s": frame_dt_s,
        "distance_bin_index": distance_bin_index,
        "occupancy_s": occupancy_s,
        "angle_edges_deg": angle_edges_deg,
        "angle_centers_deg": angle_centers_deg,
        "distance_edges_px": distance_edges_px,
        "distance_centers_px": distance_centers_px,
    }


def spike_frame_counts(spike_times, frame_times, max_lag_s):
    spike_times = np.asarray(spike_times, dtype=float)
    insertion = np.searchsorted(frame_times, spike_times)
    right = np.clip(insertion, 0, len(frame_times) - 1)
    left = np.clip(insertion - 1, 0, len(frame_times) - 1)
    use_right = (
        np.abs(frame_times[right] - spike_times)
        < np.abs(frame_times[left] - spike_times)
    )
    nearest = np.where(use_right, right, left)
    keep = np.abs(frame_times[nearest] - spike_times) <= max_lag_s
    return np.bincount(
        nearest[keep],
        minlength=len(frame_times),
    ).astype(float)


def egocentric_spike_count_map(
    event_frame_index,
    event_weights,
    distance_bin_index,
    num_distance_bins,
):
    spike_counts = np.zeros(
        (num_distance_bins, distance_bin_index.shape[1]),
        dtype=float,
    )
    for angle_index in range(distance_bin_index.shape[1]):
        bins = distance_bin_index[event_frame_index, angle_index]
        valid = bins >= 0
        spike_counts[:, angle_index] = np.bincount(
            bins[valid],
            weights=event_weights[valid],
            minlength=num_distance_bins,
        )
    return spike_counts


def smooth_egocentric_map(
    spike_counts,
    occupancy_s,
    sigma_bins,
    min_occupancy_s,
):
    numerator = gaussian_filter(
        np.asarray(spike_counts, dtype=float),
        sigma=sigma_bins,
        mode=("nearest", "wrap"),
    )
    denominator = gaussian_filter(
        np.asarray(occupancy_s, dtype=float),
        sigma=sigma_bins,
        mode=("nearest", "wrap"),
    )
    rate_map = np.full_like(numerator, np.nan)
    np.divide(
        numerator,
        denominator,
        out=rate_map,
        where=denominator > 0,
    )
    rate_map[denominator < min_occupancy_s] = np.nan
    return rate_map


def egocentric_metrics(
    rate_map,
    angle_centers_deg,
    distance_centers_px,
):
    if not np.isfinite(rate_map).any() or np.nanmax(rate_map) <= 0:
        return np.nan, np.nan, np.nan
    distance_index, angle_index = np.unravel_index(
        np.nanargmax(rate_map),
        rate_map.shape,
    )
    tuning_curve = np.nan_to_num(
        rate_map[distance_index],
        nan=0.0,
    )
    weight_sum = tuning_curve.sum()
    if weight_sum <= 0:
        rayleigh = np.nan
    else:
        vector = np.sum(
            tuning_curve
            * np.exp(1j * np.deg2rad(angle_centers_deg))
        )
        rayleigh = np.abs(vector) / weight_sum
    return (
        rayleigh,
        angle_centers_deg[angle_index],
        distance_centers_px[distance_index],
    )


def random_circular_shift(num_frames, min_shift_frames, rng):
    minimum = max(1, int(np.ceil(min_shift_frames)))
    if 2 * minimum >= num_frames:
        minimum = 1
    return int(rng.integers(minimum, num_frames - minimum + 1))


## Run per-session analysis

EBC shuffle files are cached separately from the existing allocentric spatial
information shuffles. EBC classification is independent of `place_cell`,
because a pure EBC need not have allocentric place tuning.


In [8]:
def analyze_session(num_of_rec):
    data_dir, kilosort_dir = session_paths(num_of_rec)
    (
        summary_path,
        spatial_shuffle_path,
        egocentric_shuffle_path,
        map_cache_path,
    ) = result_paths(num_of_rec)

    session_info = read_formatted_json(
        data_dir / f"{session_info_filename}.json"
    )["session_info"]
    interval_table = pd.read_csv(
        data_dir / f"{interval_table_filename}.csv"
    )
    interval_pairs_all = np.asarray(
        get_interval_pairs(interval_table, phase_key=phase_key),
        dtype=float,
    )

    exposure_timestamps, adc_time_origin_s, _ = get_exposure_timestamps(
        session_info=session_info,
        data_dir=data_dir,
    )

    pose_filename, pose_format = poseConfig[num_of_rec]
    pose = read_pose(data_dir / pose_filename, pose_format)
    pose_frames = pose["frame"].to_numpy(dtype=int)
    if pose_frames.min() < 0 or pose_frames.max() >= len(exposure_timestamps):
        raise IndexError("pose frame index is outside camera exposure timestamps")
    pose_times = exposure_timestamps[pose_frames]

    start, end = interval_pairs_all[0]
    start = max(start, pose_times[0])
    end = min(end, pose_times[-1])
    interval_pairs = np.asarray([[start, end]])
    time_support = convert_time_list_to_nap_tsd(interval_pairs)
    position = nap.TsdFrame(
        t=pose_times,
        d=pose[["center_x", "center_y"]].to_numpy(),
        columns=["x", "y"],
        time_support=time_support,
    )

    spike_times = np.load(
        data_dir / f"probe{probe_name}" / "adc_spike_time.npy",
        mmap_mode="r",
    ).reshape(-1) - adc_time_origin_s
    spike_clusters = np.load(
        kilosort_dir / "spike_clusters.npy"
    ).reshape(-1)
    cluster_KSLabel = pd.read_csv(
        kilosort_dir / "cluster_KSLabel.tsv",
        sep="	",
    )
    good_unit_ids = cluster_KSLabel.loc[
        cluster_KSLabel["KSLabel"] == "good",
        "cluster_id",
    ].to_numpy(dtype=int)
    located_spikes = locate_spikes(
        good_unit_ids,
        spike_clusters,
        spike_times,
    )
    spikes_dict = {}
    for unit_id, _, times in located_spikes:
        unit_spikes = nap.Ts(
            t=np.asarray(times, dtype=float),
            time_support=time_support,
        )
        if len(unit_spikes) >= min_spikes:
            spikes_dict[unit_id] = unit_spikes
    spikes = nap.TsGroup(spikes_dict, time_support=time_support)

    spatial_maps = nap.compute_tuning_curves(
        data=spikes,
        features=position,
        bins=spatial_bins,
        range=arena_range,
        epochs=time_support,
        feature_names=["x", "y"],
    )
    spatial_maps_smooth = smooth_rate_maps(
        spatial_maps,
        smooth_sigma,
        min_occupancy_s,
    )
    spatial_information = nap.compute_mutual_information(spatial_maps)
    unit_ids = spatial_maps.coords["unit"].values.astype(int)

    midpoint = (start + end) / 2
    half_epochs = [
        nap.IntervalSet(start=start, end=midpoint),
        nap.IntervalSet(start=midpoint, end=end),
    ]
    half_maps = [
        smooth_rate_maps(
            nap.compute_tuning_curves(
                data=spikes,
                features=position,
                bins=spatial_bins,
                range=arena_range,
                epochs=epoch,
                feature_names=["x", "y"],
            ),
            smooth_sigma,
            min_occupancy_s,
        )
        for epoch in half_epochs
    ]

    map_stability = {}
    grid_scores = {}
    border_scores = {}
    border_walls = {}
    autocorrelations = {}

    for unit_id in unit_ids:
        rate_map = spatial_maps_smooth.sel(unit=unit_id).values.T
        first_half = half_maps[0].sel(unit=unit_id).values.T
        second_half = half_maps[1].sel(unit=unit_id).values.T
        autocorrelation = spatial_autocorrelation(rate_map)
        border_scores[unit_id], border_walls[unit_id] = border_score(
            rate_map
        )
        map_stability[unit_id] = map_correlation(first_half, second_half)
        grid_scores[unit_id] = grid_score(autocorrelation)
        autocorrelations[unit_id] = autocorrelation

    if spatial_shuffle_path.exists():
        with np.load(
            spatial_shuffle_path,
            allow_pickle=False,
        ) as shuffle_result:
            null_information = pd.DataFrame(
                shuffle_result["null_information"],
                index=shuffle_result["unit_ids"],
            ).loc[unit_ids].to_numpy()
    else:
        np.random.seed(shuffle_seed)
        null_information = np.empty((len(unit_ids), num_shuffle))

        for shuffle_index in tqdm(
            range(num_shuffle),
            desc=f"spatial rec {num_of_rec}",
        ):
            shuffled_spikes = nap.shift_timestamps(
                spikes,
                min_shift=min_shift_s,
                mode="wrap",
            )
            shuffled_maps = nap.compute_tuning_curves(
                data=shuffled_spikes,
                features=position,
                bins=spatial_bins,
                range=arena_range,
                epochs=time_support,
                feature_names=["x", "y"],
            )
            null_information[:, shuffle_index] = (
                nap.compute_mutual_information(shuffled_maps)
                .loc[unit_ids, "bits/spike"]
            )

        np.savez_compressed(
            spatial_shuffle_path,
            unit_ids=unit_ids,
            null_information=null_information,
        )

    observed_information = spatial_information.loc[
        unit_ids,
        "bits/spike",
    ].to_numpy()
    spatial_p = (
        1
        + np.sum(
            null_information >= observed_information[:, None],
            axis=1,
        )
    ) / (null_information.shape[1] + 1)

    geometry = prepare_egocentric_geometry(
        pose_times=pose_times,
        position_xy=pose[["center_x", "center_y"]].to_numpy(),
        head_direction_deg=pose["hd_deg"].to_numpy(),
        interval_pair=interval_pairs[0],
        arena_range=arena_range,
        angle_bin_deg=egocentric_angle_bin_deg,
        num_distance_bins=egocentric_num_distance_bins,
    )
    frame_times = geometry["frame_times"]
    frame_dt_s = geometry["frame_dt_s"]
    distance_bin_index = geometry["distance_bin_index"]
    egocentric_occupancy_s = geometry["occupancy_s"]
    max_pose_lag_s = max(2 * frame_dt_s, 0.05)

    unit_frame_counts = []
    egocentric_spike_counts = []
    egocentric_rate_maps = []
    egocentric_rayleigh = []
    egocentric_preferred_angle_deg = []
    egocentric_preferred_distance_px = []
    egocentric_stability = []
    first_half_maps = []
    second_half_occupancy = None
    split_frame = int(np.searchsorted(frame_times, midpoint))
    first_frame_index = np.arange(split_frame)
    second_frame_index = np.arange(split_frame, len(frame_times))
    first_half_occupancy = egocentric_occupancy(
        distance_bin_index[first_frame_index],
        frame_dt_s,
        egocentric_num_distance_bins,
    )
    second_half_occupancy = egocentric_occupancy(
        distance_bin_index[second_frame_index],
        frame_dt_s,
        egocentric_num_distance_bins,
    )

    for unit_id in unit_ids:
        frame_counts = spike_frame_counts(
            spikes[unit_id].times(),
            frame_times,
            max_lag_s=max_pose_lag_s,
        )
        event_frame_index = np.flatnonzero(frame_counts)
        event_weights = frame_counts[event_frame_index]
        spike_count_map = egocentric_spike_count_map(
            event_frame_index,
            event_weights,
            distance_bin_index,
            egocentric_num_distance_bins,
        )
        rate_map = smooth_egocentric_map(
            spike_count_map,
            egocentric_occupancy_s,
            egocentric_smooth_sigma_bins,
            egocentric_min_occupancy_s,
        )
        rayleigh, preferred_angle, preferred_distance = (
            egocentric_metrics(
                rate_map,
                geometry["angle_centers_deg"],
                geometry["distance_centers_px"],
            )
        )

        first_events = event_frame_index[event_frame_index < split_frame]
        first_weights = frame_counts[first_events]
        second_events = event_frame_index[event_frame_index >= split_frame]
        second_weights = frame_counts[second_events]
        first_count_map = egocentric_spike_count_map(
            first_events,
            first_weights,
            distance_bin_index,
            egocentric_num_distance_bins,
        )
        second_count_map = egocentric_spike_count_map(
            second_events,
            second_weights,
            distance_bin_index,
            egocentric_num_distance_bins,
        )
        first_rate_map = smooth_egocentric_map(
            first_count_map,
            first_half_occupancy,
            egocentric_smooth_sigma_bins,
            egocentric_min_occupancy_s,
        )
        second_rate_map = smooth_egocentric_map(
            second_count_map,
            second_half_occupancy,
            egocentric_smooth_sigma_bins,
            egocentric_min_occupancy_s,
        )

        unit_frame_counts.append(frame_counts)
        egocentric_spike_counts.append(spike_count_map)
        egocentric_rate_maps.append(rate_map)
        egocentric_rayleigh.append(rayleigh)
        egocentric_preferred_angle_deg.append(preferred_angle)
        egocentric_preferred_distance_px.append(preferred_distance)
        egocentric_stability.append(
            map_correlation(first_rate_map, second_rate_map)
        )
        first_half_maps.append(first_rate_map)

    unit_frame_counts = np.stack(unit_frame_counts)
    egocentric_spike_counts = np.stack(egocentric_spike_counts)
    egocentric_rate_maps = np.stack(egocentric_rate_maps)
    egocentric_rayleigh = np.asarray(egocentric_rayleigh)
    egocentric_stability = np.asarray(egocentric_stability)

    use_cached_egocentric_shuffle = False
    if egocentric_shuffle_path.exists():
        with np.load(
            egocentric_shuffle_path,
            allow_pickle=False,
        ) as shuffle_result:
            required_keys = {
                "unit_ids",
                "null_rayleigh",
                "null_stability",
            }
            if required_keys.issubset(shuffle_result.files):
                cached_ids = shuffle_result["unit_ids"].astype(int)
                if set(unit_ids).issubset(cached_ids):
                    index = pd.Index(cached_ids).get_indexer(unit_ids)
                    null_egocentric_rayleigh = shuffle_result[
                        "null_rayleigh"
                    ][index]
                    null_egocentric_stability = shuffle_result[
                        "null_stability"
                    ][index]
                    use_cached_egocentric_shuffle = (
                        null_egocentric_rayleigh.shape
                        == (len(unit_ids), egocentric_num_shuffle)
                        and null_egocentric_stability.shape
                        == (len(unit_ids), egocentric_num_shuffle)
                    )

    if not use_cached_egocentric_shuffle:
        rng = np.random.default_rng(
            egocentric_shuffle_seed + int(num_of_rec)
        )
        null_egocentric_rayleigh = np.empty(
            (len(unit_ids), egocentric_num_shuffle),
            dtype=float,
        )
        null_egocentric_stability = np.empty_like(
            null_egocentric_rayleigh
        )
        min_shift_frames = egocentric_min_shift_s / frame_dt_s

        for unit_index, unit_id in enumerate(
            tqdm(
                unit_ids,
                desc=f"egocentric rec {num_of_rec}",
            )
        ):
            frame_counts = unit_frame_counts[unit_index]
            event_frame_index = np.flatnonzero(frame_counts)
            event_weights = frame_counts[event_frame_index]
            second_events = event_frame_index[
                event_frame_index >= split_frame
            ]
            second_weights = frame_counts[second_events]
            second_length = len(frame_times) - split_frame

            for shuffle_index in range(egocentric_num_shuffle):
                shift = random_circular_shift(
                    len(frame_times),
                    min_shift_frames,
                    rng,
                )
                shifted_events = (
                    event_frame_index + shift
                ) % len(frame_times)
                shuffled_count_map = egocentric_spike_count_map(
                    shifted_events,
                    event_weights,
                    distance_bin_index,
                    egocentric_num_distance_bins,
                )
                shuffled_rate_map = smooth_egocentric_map(
                    shuffled_count_map,
                    egocentric_occupancy_s,
                    egocentric_smooth_sigma_bins,
                    egocentric_min_occupancy_s,
                )
                null_egocentric_rayleigh[
                    unit_index,
                    shuffle_index,
                ] = egocentric_metrics(
                    shuffled_rate_map,
                    geometry["angle_centers_deg"],
                    geometry["distance_centers_px"],
                )[0]

                second_shift = random_circular_shift(
                    second_length,
                    min_shift_frames,
                    rng,
                )
                shifted_second_events = (
                    (second_events - split_frame + second_shift)
                    % second_length
                ) + split_frame
                shuffled_second_count_map = (
                    egocentric_spike_count_map(
                        shifted_second_events,
                        second_weights,
                        distance_bin_index,
                        egocentric_num_distance_bins,
                    )
                )
                shuffled_second_rate_map = smooth_egocentric_map(
                    shuffled_second_count_map,
                    second_half_occupancy,
                    egocentric_smooth_sigma_bins,
                    egocentric_min_occupancy_s,
                )
                null_egocentric_stability[
                    unit_index,
                    shuffle_index,
                ] = map_correlation(
                    first_half_maps[unit_index],
                    shuffled_second_rate_map,
                )

        np.savez_compressed(
            egocentric_shuffle_path,
            unit_ids=unit_ids,
            null_rayleigh=null_egocentric_rayleigh,
            null_stability=null_egocentric_stability,
        )

    valid_rayleigh_null = np.isfinite(null_egocentric_rayleigh)
    egocentric_p = (
        1
        + np.sum(
            valid_rayleigh_null
            & (
                null_egocentric_rayleigh
                >= egocentric_rayleigh[:, None]
            ),
            axis=1,
        )
    ) / (valid_rayleigh_null.sum(axis=1) + 1)
    egocentric_p[~np.isfinite(egocentric_rayleigh)] = np.nan

    valid_stability_null = np.isfinite(null_egocentric_stability)
    egocentric_stability_p = (
        1
        + np.sum(
            valid_stability_null
            & (
                null_egocentric_stability
                >= egocentric_stability[:, None]
            ),
            axis=1,
        )
    ) / (valid_stability_null.sum(axis=1) + 1)
    egocentric_stability_p[~np.isfinite(egocentric_stability)] = np.nan

    summary = spatial_information.loc[unit_ids].copy()
    summary["mean_rate_hz"] = spatial_maps.attrs["rates"]
    summary["peak_rate_hz"] = np.nanmax(
        spatial_maps_smooth.values,
        axis=(1, 2),
    )
    summary["map_stability"] = pd.Series(map_stability)
    summary["grid_score"] = pd.Series(grid_scores)
    summary["border_score"] = pd.Series(border_scores)
    summary["border_wall"] = pd.Series(border_walls)
    summary["spatial_p"] = spatial_p
    summary["egocentric_rayleigh"] = egocentric_rayleigh
    summary["egocentric_preferred_angle_deg"] = (
        egocentric_preferred_angle_deg
    )
    summary["egocentric_preferred_distance_px"] = (
        egocentric_preferred_distance_px
    )
    summary["egocentric_p"] = egocentric_p
    summary["egocentric_stability"] = egocentric_stability
    summary["egocentric_stability_p"] = egocentric_stability_p
    presence_edges = np.arange(start, end + presence_bin_s, presence_bin_s)
    summary["presence_ratio"] = [
        np.mean(
            np.histogram(spikes[unit_id].times(), presence_edges)[0] > 0
        )
        for unit_id in unit_ids
    ]
    summary["spike_time_coverage"] = [
        (spikes[unit_id].times()[-1] - spikes[unit_id].times()[0])
        / (end - start)
        for unit_id in unit_ids
    ]
    summary["temporal_qc"] = (
        (summary["presence_ratio"] >= min_presence_ratio)
        & (
            summary["spike_time_coverage"]
            >= min_spike_time_coverage
        )
    )
    summary["place_cell"] = (
        summary["temporal_qc"]
        & (summary["spatial_p"] <= spatial_alpha)
        & (summary["map_stability"] >= min_map_stability)
    )
    summary["grid_cell"] = (
        summary["place_cell"]
        & (summary["grid_score"] >= min_grid_score)
    )
    summary["boundary_cell"] = (
        summary["place_cell"]
        & (summary["border_score"] >= min_border_score)
    )
    summary["egocentric_boundary_cell"] = (
        summary["temporal_qc"]
        & (summary["egocentric_p"] <= egocentric_alpha)
    )
    summary["egocentric_stable"] = (
        summary["egocentric_stability_p"] <= egocentric_alpha
    )
    summary["stable_egocentric_boundary_cell"] = (
        summary["egocentric_boundary_cell"]
        & summary["egocentric_stable"]
    )
    summary = summary.sort_values("bits/spike", ascending=False)

    spike_positions = [
        spikes[unit_id].value_from(position).values
        for unit_id in unit_ids
    ]
    spike_position_offsets = np.r_[
        0,
        np.cumsum([len(values) for values in spike_positions]),
    ]

    summary.to_csv(summary_path)
    np.savez_compressed(
        map_cache_path,
        unit_ids=unit_ids,
        rate_maps=spatial_maps.values,
        rate_maps_smooth=spatial_maps_smooth.values,
        occupancy_s=(
            np.asarray(spatial_maps.attrs["occupancy"]).T
            / spatial_maps.attrs["fs"]
        ),
        autocorrelations=np.stack(
            [autocorrelations[unit_id] for unit_id in unit_ids]
        ),
        position=pose[["center_x", "center_y"]].to_numpy(),
        head_direction_deg=pose["hd_deg"].to_numpy(),
        position_frames=pose_frames,
        position_times=pose_times,
        spike_positions=np.vstack(spike_positions),
        spike_position_offsets=spike_position_offsets,
        arena_range=np.asarray(arena_range, dtype=float),
        interval_pairs=interval_pairs,
        egocentric_spike_counts=egocentric_spike_counts,
        egocentric_rate_maps=egocentric_rate_maps,
        egocentric_occupancy_s=egocentric_occupancy_s,
        egocentric_angle_edges_deg=geometry["angle_edges_deg"],
        egocentric_angle_centers_deg=geometry["angle_centers_deg"],
        egocentric_distance_edges_px=geometry["distance_edges_px"],
        egocentric_distance_centers_px=geometry[
            "distance_centers_px"
        ],
        egocentric_frame_times=frame_times,
    )

    print(
        f"rec {num_of_rec}: {len(position)} frames, {len(unit_ids)} units, "
        f"{summary['egocentric_boundary_cell'].sum()} EBCs -> "
        f"{map_cache_path.name}"
    )
    return (
        summary_path,
        spatial_shuffle_path,
        egocentric_shuffle_path,
        map_cache_path,
    )


In [9]:
for num_of_rec in analyzeRecList:
    analyze_session(num_of_rec)


egocentric rec 11: 100%|██████████| 44/44 [00:16<00:00,  2.72it/s]


rec 11: 28426 frames, 44 units, 13 EBCs -> spatial_maps_rec11_baseline_ProbeA.npz


egocentric rec 12: 100%|██████████| 44/44 [00:29<00:00,  1.47it/s]


rec 12: 49448 frames, 44 units, 31 EBCs -> spatial_maps_rec12_baseline_ProbeA.npz
